# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint

Automated facial expression recognition provides an objective assessment of emotions. Human based assessment of emotions has many limitations and biases and automated facial expression technology has been found to deliver a better level of insight into behavior patterns. Emotion detection from facial expressions using AI is useful in automatically measuring consumers’ engagement with their content and brands, audience engagement for advertisements, customer satisfaction in the retail sector, psychological analyses, law enforcement etc.

**Objectives:**

**Stage 4 (20 Marks):** Train a CNN Model, update your HuggingFace Space repository, and test your Deployment for Expression Recognition on the HuggingFace Space App.

##**Stage 4 (20 Marks)**

**(i) Train a CNN Model for Expression Recognition on given Expression data**

**(ii) Deploy the Model and Perform Expression Recognition on Team Data through the HuggingFace Space App**


---


* Define and train a CNN for expression recognition for the data under folder `"Expression_data"` which segregated on expression basis.

* Collect your team data by running the code cells provided within the notebook below.

* Test your model on the collected team data and optimize the CNN architecture for predicting the respective labels of the images.

- Update files present in your cloned HuggingFace Space repository.

    - Save and Download the trained expression model (`expression_model.t7`) and upload/place it in your HuggingFace Space repository within **`app/Hackathon_setup/`** folder.
    
    - Update the model architecture in the **`app/Hackathon_setup/exp_recognition_model.py`** file.
    
    - Update the code in the **`get_expression()`** function of the **`app/Hackathon_setup/exp_recognition.py`** file. (See Deployment related files)

- Commit your changes and push to HuggingFace Space repository.

- Access the `App` tab of your repository to see the build progress (debug if error persists) [This step might take sometime.]

- Once the app has built successfully, you should see below message

    `Application startup complete. Uvicorn running on http://0.0.0.0:8001`

- Test the model's Expression Recognition functionality using the application running on your Space

    Go to 3-dots icon beside Settings, then select `Embed this Space` option, and go to `Direct URL`
    - Select the task, `Expression Recognition`
    - Select 'Send Anyway' when prompted
    - Upload your image and test
    - **NOTE:** When using the Direct URL link via android mobile, the camera option will also enable to capture images (Set 1:1 aspect ratio in camera settings before-hand)


### **Download the dataset**

In [1]:
#@title Run this cell to download the dataset

from IPython import get_ipython
ipython = get_ipython()

notebook="M3_Hackathon" #name of the notebook

def setup():
#  ipython.magic("sx pip3 install torch")
    ipython.magic("sx wget wget https://cdn.talentsprint.com/aiml/Experiment_related_data/Expression_data.zip")
    ipython.magic("sx unzip Expression_data.zip")
    ipython.magic("sx wget https://cdn.iisc.talentsprint.com/AIandMLOps/Datasets/lbpcascade_frontalface.xml")

    print ("Setup completed successfully")
    return
setup()

Setup completed successfully


In [2]:
%ls

Expression_data/     lbpcascade_frontalface.xml  sample_data/
Expression_data.zip  __MACOSX/


**Dataset attributes:**

During the setup you have downloaded the `Expression_data`:

* **Expression_data**: In this folder, the images are segregrated in terms of Expression
> * Expressions available: ANGER, DISGUST, FEAR, HAPPINESS, NEUTRAL, SADNESS, SURPRISE
> * Each class is organised as one folder
> * There are ~18000 total images in the training data and ~4500 total images in the testing data

### **Import Required Packages**

In [3]:
%matplotlib inline
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
from torch.utils.data import DataLoader,Dataset
import matplotlib.pyplot as plt
import torchvision.utils
import numpy as np
import random
from PIL import Image                   # PIL (Pillow) is the Python Image Library. Used to cut and resize images, or do simple manipulation.
import torch
from torch.autograd import Variable
import PIL.ImageOps
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import os
import warnings
from time import sleep
import sys
warnings.filterwarnings('ignore')

For the following step, to obtain hints on building a CNN model for face expression, you may refer to this [article](https://drive.google.com/open?id=1P2rpaWW3tOtGGnw4dvtdZ4hjoc8iDNst).

**Define and train a CNN model for expression recognition**

In [4]:
#YOUR CODE HERE : Sample Helper function
train_dir = './Expression_data/Facial_expression_train'
test_dir = './Expression_data/Facial_expression_test'

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# The dataset images are colour; the model works on single-channel (grayscale) faces
def rgb2gray(image):
    return image.convert('L')

# Transform for validation, team data and deployment.
# The SAME transform is used in exp_recognition_model.py on the server.
trnscm = transforms.Compose([rgb2gray, transforms.Resize((48,48)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# Training-only transform: random flips, small rotations/shifts and lighting changes,
# so the model learns the expression rather than memorising particular images
train_transform = transforms.Compose([
    rgb2gray,
    transforms.Resize((48,48)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])


In [4]:
train_dataset = dset.ImageFolder(root=train_dir, transform=train_transform)
test_dataset = dset.ImageFolder(root=test_dir, transform=trnscm)

# Index -> expression name. ImageFolder numbers the folders alphabetically,
# which matches the classes dictionary in exp_recognition_model.py
classes = {idx: name for name, idx in train_dataset.class_to_idx.items()}
print(classes)
assert classes == {0: 'ANGER', 1: 'DISGUST', 2: 'FEAR', 3: 'HAPPINESS', 4: 'NEUTRAL', 5: 'SADNESS', 6: 'SURPRISE'}

num_workers = 2
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=num_workers, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=num_workers)

In [5]:
#YOUR CODE HERE : Check number of training and Validation images
print("Training images:  ", len(train_dataset))
print("Validation images:", len(test_dataset))
print()
train_counts = np.bincount(train_dataset.targets, minlength=len(classes))
test_counts = np.bincount(test_dataset.targets, minlength=len(classes))
for idx, name in classes.items():
    print(f"{name:<10} train {train_counts[idx]:5d}   validation {test_counts[idx]:5d}")


In [6]:
#YOUR CODE HERE : Generate a batch of 10 images and labels
images, labels = next(iter(DataLoader(train_dataset, batch_size=10, shuffle=True)))
fig, axes = plt.subplots(1, 10, figsize=(20, 3))
for ax, img, label in zip(axes, images, labels):
    ax.imshow(img.squeeze() * 0.5 + 0.5, cmap='gray')   # undo Normalize for display
    ax.set_title(classes[label.item()])
    ax.axis('off')
plt.show()


In [ ]:
# CNN for 48x48 grayscale faces. Four conv blocks (48 -> 24 -> 12 -> 6 -> 3 pixels), then a small classifier.
# This exact class is copied into exp_recognition_model.py so the saved weights load on the server.
class facExpRec(nn.Module):
    def __init__(self, num_classes=7):
        super(facExpRec, self).__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Dropout(0.1),
            )

        self.features = nn.Sequential(conv_block(1, 32), conv_block(32, 64), conv_block(64, 128), conv_block(128, 256))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
# YOUR CODE HERE : Print the summary of the model
model = facExpRec().to(device)
print(model)
print("\nTrainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
with torch.no_grad():
    print("Output shape for a batch of 2 faces:", tuple(model.eval()(torch.zeros(2, 1, 48, 48, device=device)).shape))

In [ ]:
# Train the CNN. Takes roughly 20-30 minutes on a Colab T4 GPU.
import copy

# Rarer expressions (e.g. FEAR has ~450 images vs ~4800 for HAPPINESS) get a larger weight in the loss,
# so the model does not learn to simply ignore them
class_weights = torch.tensor((train_counts.sum() / train_counts) ** 0.5, dtype=torch.float32)
class_weights = class_weights / class_weights.mean()
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

num_epochs = 80
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, epochs=num_epochs, steps_per_epoch=len(train_loader))

def evaluate(net, loader):
    net.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            preds.append(net(x.to(device)).argmax(1).cpu())
            labels.append(y)
    preds, labels = torch.cat(preds), torch.cat(labels)
    return (preds == labels).float().mean().item(), preds, labels

history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
best_acc, best_state = 0.0, None
for epoch in range(num_epochs):
    model.train()
    total_loss, correct, seen = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        seen += y.size(0)

    val_acc, _, _ = evaluate(model, test_loader)
    history['train_loss'].append(total_loss / seen)
    history['train_acc'].append(correct / seen)
    history['val_acc'].append(val_acc)
    if val_acc > best_acc:
        best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch+1:2d}/{num_epochs}  loss {total_loss/seen:.3f}  train acc {correct/seen:.3f}  validation acc {val_acc:.3f}")

# Keep the weights from the epoch with the best validation accuracy
model.load_state_dict(best_state)
print("\nBest validation accuracy: %.3f" % best_acc)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss']); ax1.set_title('Training loss'); ax1.set_xlabel('epoch')
ax2.plot(history['train_acc'], label='train'); ax2.plot(history['val_acc'], label='validation')
ax2.set_title('Accuracy'); ax2.set_xlabel('epoch'); ax2.legend()
plt.show()

**Test your model and optimize CNN architecture for predicting the labels correctly**

In [ ]:
# YOUR CODE HERE for test evaluation
test_acc, test_preds, test_labels = evaluate(model, test_loader)
print("Overall validation accuracy: %.3f" % test_acc)
print()
for idx, name in classes.items():
    mask = test_labels == idx
    print(f"{name:<10} {(test_preds[mask] == idx).float().mean().item():.3f}   ({mask.sum().item()} images)")

In [ ]:
# Confusion matrix: rows are the true expression, columns the predicted one.
# The diagonal is correct predictions; off-diagonal cells show which expressions get mixed up.
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(test_labels.numpy(), test_preds.numpy(),
                                        display_labels=[classes[i] for i in range(len(classes))],
                                        normalize='true', values_format='.2f', cmap='Blues', xticks_rotation=45)
plt.title("Validation confusion matrix (fraction of each true class)")
plt.show()

#### **Team Data Collection**

**Collect your team data and fine-tune the CNN for expression data on your team**

- Collect Team Data by running the code cells provided below
    - The collected Expression images of your team will be stored in the `captured_images_with_Expression` directory

    - NOTE: *Since team members will be using separate colab notebooks, they can capture their own face images and then download it and share with other members for model testing.* Code cell is provided below to download the data.

- This data will be useful for testing the above trained cnn network

In [ ]:
# @title Run this cell to Setup Image Capturing in Colab {display-mode: "form"}

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import imageio
import datetime
import pathlib
import cv2
import numpy as np
import matplotlib.pyplot as plt

AREA_THRESHOLD = 2304

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)

  im = Image.open(filename)
  im1 = im.crop((80, 0, 560, 480))
  im1.save(filename)

  return filename


def save_faces(miniframe, filepath):
    TRAINSET = "lbpcascade_frontalface.xml"
    classifier = cv2.CascadeClassifier(TRAINSET)
    faces = classifier.detectMultiScale(miniframe)
    image = get_large_face(miniframe, faces)
    if not isinstance(image, np.ndarray):
        return {"status" : False}
    plt.imshow(image)
    plt.show()
    #cv2.imwrite(filepath, image)
    imageio.imwrite(filepath, image)
    return {"status" : True}

def get_large_face(miniframe, faces):
    images = []
    face_areas = []
    required_image = 0
    for x,y,w,h in faces:
        face_cropped = miniframe[y:y+h, x:x+w]
        face_areas.append(w*h)
        images.append(face_cropped)
        required_image = images[np.argmax(face_areas)]
    if not face_areas:
        return 0
    if face_areas[np.argmax(face_areas)] < AREA_THRESHOLD:
        return 0

    return required_image

def save_image(filename, class_name):
    base_path = "captured_images_with_Expression/"

    pathlib.Path(base_path + class_name).mkdir(parents=True, exist_ok=True)
    file_name = class_name + "_" + datetime.datetime.now().strftime("%s") + ".jpg"
    filepath = base_path +class_name + "/" + file_name

    image = Image.open(filename)
    miniframe = np.asarray(image)
    status = save_faces(miniframe, filepath)
    if status['status']:
        print("Image saved in " + filepath, flush = True)
    else:
        print("Face not found!\nRetry!")


In [ ]:
# @title Capture an Image for ANGER $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "ANGER"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for DISGUST $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "DISGUST"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for FEAR $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "FEAR"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for HAPPINESS $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "HAPPINESS"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for NEUTRAL $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "NEUTRAL"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for SADNESS $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "SADNESS"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for SURPRISE $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "SURPRISE"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Download Collected Images $ $ [OPTIONAL]  {display-mode: "form"}

from google.colab import files
!zip -r "captured_images_with_Expression.zip" "captured_images_with_Expression"
files.download('captured_images_with_Expression.zip')
print("Downloaded captured_images_with_Expression.zip !!")

In [ ]:
%ls

**While uploading the team images manually to captured_images. this file .ipynb_checkpoints will be created and make issue to delet run the below code**

In [ ]:
rm -rf `find -type d -name .ipynb_checkpoints`

In [ ]:
# YOUR CODE HERE for loading the team expression data. Note: Use the same transform which used for Expression_Data.
# YOU CODE HERE for Dataloader
team_dir = './captured_images_with_Expression'

# Labels come from the folder NAME, using the same numbering as the training data.
# (ImageFolder would renumber the classes if your team skipped an expression, giving wrong labels.)
class_to_idx = {name: idx for idx, name in classes.items()}
team_samples = []
for name in sorted(os.listdir(team_dir)):
    folder = os.path.join(team_dir, name)
    if os.path.isdir(folder) and name in class_to_idx:
        for f in sorted(os.listdir(folder)):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                team_samples.append((os.path.join(folder, f), class_to_idx[name]))

# The app crops every uploaded photo with detected_face() before predicting. To test (and fine-tune) on the same kind
# of image, apply the identical crop: largest face found by the same Haar cascade, on the same grayscale conversion,
# falling back to the whole image when no face is found.
import cv2
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def app_face_crop(img):
    gray = cv2.cvtColor(np.array(img.convert('RGB')), cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    if len(faces) == 0:
        return Image.fromarray(gray)
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    return Image.fromarray(gray[y:y+h, x:x+w])

class TeamExpressionDataset(Dataset):
    def __init__(self, samples, transform, return_path=True):
        self.samples = samples
        self.transform = transform
        self.return_path = return_path

    def __getitem__(self, index):
        path, label = self.samples[index]
        image = self.transform(app_face_crop(Image.open(path)))
        return (image, label, path) if self.return_path else (image, label)

    def __len__(self):
        return len(self.samples)

team_dataset = TeamExpressionDataset(team_samples, trnscm)
team_loader = DataLoader(team_dataset, batch_size=32, shuffle=False)

print("Team images:", len(team_dataset))
team_counts = np.bincount([label for _, label in team_samples], minlength=len(classes))
for idx, name in classes.items():
    print(f"{name:<10} {team_counts[idx]}")

In [ ]:
# YOUR CODE HERE for getting the CNN representation of your team data with expression. Optimize the CNN model for predicting the labels of expressions correctly
# Note: If the CNN Model is not performing as expected, then you can add your Team Data to the Existing Training Data and Re-Train the Model.
def predict_team(net):
    net.eval()
    rows = []
    with torch.no_grad():
        for x, y, paths in team_loader:
            preds = net(x.to(device)).argmax(1).cpu()
            rows += list(zip(paths, y.tolist(), preds.tolist()))
    return rows

team_rows = predict_team(model)
print("Team data accuracy: %.3f (%d of %d correct)" % (np.mean([t == p for _, t, p in team_rows]),
                                                       sum(t == p for _, t, p in team_rows), len(team_rows)))

# Show every team image with its true and predicted expression (green = correct, red = wrong)
cols = 6
rows_n = int(np.ceil(len(team_rows) / cols))
fig, axes = plt.subplots(rows_n, cols, figsize=(3 * cols, 3.3 * rows_n), squeeze=False)
for ax in axes.flat:
    ax.axis('off')
for ax, (path, true, pred) in zip(axes.flat, team_rows):
    ax.imshow(Image.open(path).convert('L'), cmap='gray')
    ax.set_title(f"true: {classes[true]}\npred: {classes[pred]}", color='green' if true == pred else 'red')
plt.tight_layout()
plt.show()

In [ ]:
# OPTIONAL fine-tuning: run this only if the model gets too many of your team's expressions wrong above.
# It retrains briefly on the original training data plus your team photos (repeated so they carry enough weight),
# at a low learning rate so the model adapts to your faces without forgetting the rest.
# NOTE: after this, the team predictions are no longer an honest test (the model has seen those photos).
#       Capture a few NEW photos to check it, or test on the deployed app.
team_repeats = 20
finetune_epochs = 5

team_train = TeamExpressionDataset(team_samples, train_transform, return_path=False)
finetune_dataset = torch.utils.data.ConcatDataset([train_dataset] + [team_train] * team_repeats)
finetune_loader = DataLoader(finetune_dataset, batch_size=64, shuffle=True, num_workers=num_workers, drop_last=True)
finetune_optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

for epoch in range(finetune_epochs):
    model.train()
    for x, y in finetune_loader:
        x, y = x.to(device), y.to(device)
        finetune_optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        finetune_optimizer.step()
    val_acc, _, _ = evaluate(model, test_loader)
    team_acc = np.mean([t == p for _, t, p in predict_team(model)])
    print(f"Fine-tune epoch {epoch+1}/{finetune_epochs}  validation acc {val_acc:.3f}  team acc {team_acc:.3f}")

**Save your trained model**

* Save the state dictionary of the classifier (use pytorch only), It will be useful in
integrating model to the mobile app

 [Hint](https://pytorch.org/tutorials/beginner/saving_loading_models.html)

In [ ]:
### YOUR CODE HERE for saving the CNN model
# Save only the state dictionary (weights), under the 'net_dict' key that get_expression() reads on the server
torch.save({'net_dict': model.state_dict()}, 'expression_model.t7')

# Check the saved file loads back into a fresh copy of the network
check_net = facExpRec()
print(check_net.load_state_dict(torch.load('expression_model.t7', map_location='cpu')['net_dict']))
print("File size: %.1f MB" % (os.path.getsize('expression_model.t7') / 1e6))

**Download your trained model**
* Given the path of model file the following code downloads it through the browser

In [ ]:
from google.colab import files
files.download('expression_model.t7')
#files.download('<model_file_path>')